<a href="https://colab.research.google.com/github/sargisssss/FitMatch/blob/main/Copy_of_Fitmatch_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import files
files.upload()

Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"sargisarakelyan","key":"KGAT_9218e956888a7fb6aa39a1a3a397bdb3"}\r\n'}

In [2]:
!mkdir -p ~/.kaggle
!mv kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [3]:
!pip install kaggle gradio torch torchvision git+https://github.com/openai/CLIP.git

  Cloning https://github.com/openai/CLIP.git to /tmp/pip-req-build-_hm62_r0
  Running command git clone --filter=blob:none --quiet https://github.com/openai/CLIP.git /tmp/pip-req-build-_hm62_r0
  Resolved https://github.com/openai/CLIP.git to commit d05afc436d78f1c48dc0dbf8e5980a9d471f35f6
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.3 MB/s eta 0:00:00
  Created wheel for clip: filename=clip-1.0-py3-none-any.whl size=1369490 sha256=a7476b09a40af32cb3a9d689e86ddfb0b589f0b685af4c8c454d0a8377d5064e
  Stored in directory: /tmp/pip-ephem-wheel-cache-gvx14lxd/wheels/35/3e/df/3d24cbfb3b6a06f17a2bfd7d1138900d4365d9028aa8f6e92f
Successfully built clip


In [4]:
!kaggle datasets download -d maparla/zara-products -p /content/zara/
!unzip /content/zara/zara-products.zip -d /content/zara/

Выходные данные были обрезаны до нескольких последних строк (5000).
  inflating: /content/zara/images/zara/cb731fe9-938f-457d-b2ba-0dd7e5679913.jpg  
  inflating: /content/zara/images/zara/cb74fbd8-f135-43bc-8995-4f09dabfffca.jpg  
  inflating: /content/zara/images/zara/cb75bbc9-31db-4463-b213-cd8abc6dabb4.jpg  
  inflating: /content/zara/images/zara/cb77d387-c404-4771-92cb-cf61329a47cf.jpg  
  inflating: /content/zara/images/zara/cb78c174-954b-4927-9dc2-9685bc456a24.jpg  
  inflating: /content/zara/images/zara/cb7ba70e-0a54-4fa5-b417-fe6bde46809d.jpg  
  inflating: /content/zara/images/zara/cb7f4a44-97c1-44ea-b647-c3d805f49e92.jpg  
  inflating: /content/zara/images/zara/cb840156-b8c7-4c7c-8c20-d2c44a31ad5c.jpg  
  inflating: /content/zara/images/zara/cb8979d6-9230-464b-9746-309ed15f89c7.jpg  
  inflating: /content/zara/images/zara/cb8bc856-fc20-4e20-938a-0b5bef26787e.jpg  
  inflating: /content/zara/images/zara/cb8d005c-ed64-4555-8d43-81aee2123b27.jpg  
  inflating: /content/zara/ima

In [14]:
import os
import ast
import pandas as pd
from PIL import Image

images_dir = '/content/zara/images/zara'
csv_path = '/content/zara/store_zara.csv'

df_raw = pd.read_csv(csv_path)

STYLE_MAPPING = {
    'jackets': ['Casual', 'Formal'], 'puffers': ['Casual', 'Formal'], 'hoodies': ['Casual', 'Formal'],
    'sweatshirts': ['Casual', 'Formal'], 't-shirts': ['Casual', 'Formal'], 'jeans': ['Casual', 'Formal'],
    'shorts': ['Casual', 'Formal'], 'overshirts': ['Casual', 'Formal'], 'linen': ['Casual', 'Formal'],
    'sweaters': ['Casual', 'Formal'], 'cardigans': ['Casual', 'Formal'], 'knitwear': ['Casual', 'Formal'],
    'dresses': ['Casual', 'Formal'], 'skirts': ['Casual', 'Formal'], 'tops': ['Casual', 'Formal'],
    'bodysuits': ['Casual', 'Formal'], 'bags': ['Casual', 'Formal'],
    'suits': ['Casual', 'Formal'], 'blazers': ['Casual', 'Formal'], 'pants': ['Casual', 'Formal'],
    'coats': ['Casual', 'Formal'], 'shoes': ['Casual', 'Formal'],
    'tracksuits': ['Casual', 'Formal'],
}

TYPE_MAPPING = {
    'jackets': 'Jackets', 'puffers': 'Jackets', 'hoodies': 'Hoodies',
    'sweatshirts': 'Sweatshirts', 't-shirts': 'Tshirts', 'jeans': 'Jeans',
    'shorts': 'Shorts', 'overshirts': 'Shirts', 'linen': 'Shirts',
    'sweaters': 'Sweaters', 'cardigans': 'Cardigans', 'knitwear': 'Knitwear',
    'dresses': 'Dresses', 'skirts': 'Skirts', 'tops': 'Tops',
    'bodysuits': 'Bodysuits', 'bags': 'Handbags', 'suits': 'Blazers',
    'blazers': 'Blazers', 'pants': 'Trousers', 'coats': 'Coats',
    'shoes': 'Casual Shoes', 'tracksuits': 'Track Pants',
}

rows = []
for _, row in df_raw.iterrows():
    try:
        image_ids = ast.literal_eval(row['image_downloads'])
    except:
        continue

    usages = STYLE_MAPPING.get(row['terms'], ['Casual', 'Formal'])
    article_type = TYPE_MAPPING.get(row['terms'], row['terms'].capitalize())
    gender = 'Men' if row['section'] == 'MAN' else 'Women'

    for img_id in image_ids:
        img_path = os.path.join(images_dir, f"{img_id}.jpg")
        if os.path.exists(img_path):
            for u in usages:
                rows.append({
                    'id': img_id,
                    'gender': gender,
                    'articleType': article_type,
                    'usage': u,
                    'productDisplayName': str(row['name']).title() if pd.notna(row['name']) else 'Unknown',
                    'price': row['price'],
                    'currency': row['currency'],
                    'terms': row['terms'],
                    'baseColour': 'Unknown',
                })

df = pd.DataFrame(rows)
print(f"Total items: {len(df)}")
print(f"\nArticle types:\n{df['articleType'].value_counts()}")
print(f"\nGender:\n{df['gender'].value_counts()}")
print(f"\nUsage:\n{df['usage'].value_counts()}")

Total items: 50110

Article types:
articleType
Tops            7420
Jackets         5966
Casual Shoes    5954
Trousers        5240
Jeans           4060
Sweaters        3298
Dresses         3010
Skirts          2852
Handbags        2638
Tshirts         1902
Sweatshirts     1620
Coats           1608
Blazers         1060
Cardigans       1060
Shorts           820
Hoodies          592
Shirts           460
Track Pants      326
Knitwear         210
Bodysuits         14
Name: count, dtype: int64

Gender:
gender
Women    32612
Men      17498
Name: count, dtype: int64

Usage:
usage
Casual    25055
Formal    25055
Name: count, dtype: int64


In [15]:
import clip
import torch
import numpy as np
from PIL import Image
from pathlib import Path

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

model, preprocess = clip.load("ViT-B/32", device=device)
model.eval()
print("CLIP loaded successfully")

STYLE_MAP = {'Casual': 'Casual', 'Sports': 'Sports', 'Formal': 'Formal'}

BLACKLIST = {
    '117140f8-0d19-4d6c-81e3-fd38022e6d15',
    'de5939f1-92d6-452d-a0b2-8a93659f7b96',
    '5fae5bc3-7965-419b-8066-50bfbf127358',
    '4420d9a8-f174-4534-bd41-2d5bd2e365e8',
    '05f06785-9a6c-43a4-a348-66ddfe1ef7bd',
    '9dacd736-aff7-424d-a5b8-b762401f7fba',
    'fee5e3bd-c574-4c5f-ac93-3718f2c7ef5d',
    'a0be7bda-c158-44bd-8024-aaa9477252e3',
    '8ebca527-c0ae-453a-b064-06443900fc77',
    '2b0e7204-8c24-4d82-aa1b-a53177bd2d17',
    'e6dd520a-23d1-4d74-88cb-87d531de0797',
    'f11463fe-2991-42e1-b014-8261134bd296'
}

df_clean = df[df['usage'].isin(STYLE_MAP.keys())].reset_index(drop=True)
df_clean = df_clean[~df_clean['id'].isin(BLACKLIST)].reset_index(drop=True)

print(f"Items after filtering: {len(df_clean)}")
print(df_clean['usage'].value_counts())

Device: cuda
CLIP loaded successfully
Items after filtering: 50086
usage
Casual    25043
Formal    25043
Name: count, dtype: int64


In [28]:
fixes = {
    'Oversize Denim T-Shirt Limited Edition': 'Tshirts',
    'Quarter Zip Denim Sweatshirt': 'Sweatshirts',
    'Denim Shirt': 'Shirts'
}

for name, correct_type in fixes.items():
    mask = df['productDisplayName'] == name
    df.loc[mask, 'articleType'] = correct_type
    print(f"Fixed {mask.sum()} items: '{name}' → {correct_type}")

df_clean = df[df['usage'].isin(STYLE_MAP.keys())].reset_index(drop=True)
df_clean = df_clean[~df_clean['id'].isin(BLACKLIST)].reset_index(drop=True)

id_to_color = dict(zip(df_clean['id'], df_clean['baseColour']))
id_to_type = dict(zip(df_clean['id'], df_clean['articleType']))
id_to_row = df_clean.set_index('id')

print(f"Items after fixes: {len(df_clean)}")

Fixed 20 items: 'Oversize Denim T-Shirt Limited Edition' → Tshirts
Fixed 18 items: 'Quarter Zip Denim Sweatshirt' → Sweatshirts
Fixed 62 items: 'Denim Shirt' → Shirts
Items after fixes: 50086


In [29]:
from google.colab import drive
drive.mount('/content/drive')

import numpy as np
all_embeddings = np.load('/content/drive/MyDrive/FitMatch/clip_embeddings.npy')
all_ids = np.load('/content/drive/MyDrive/FitMatch/clip_ids.npy')
print(f"Loaded embeddings: {all_embeddings.shape}")


# from torch.utils.data import DataLoader, Dataset
# import torch
# import numpy as np
# from pathlib import Path

# class FashionImageDataset(Dataset):
#     def __init__(self, df, images_dir, preprocess):
#         self.df = df.reset_index(drop=True)
#         self.images_dir = images_dir
#         self.preprocess = preprocess

#     def __len__(self):
#         return len(self.df)

#     def __getitem__(self, idx):
#         row = self.df.iloc[idx]
#         path = f"{self.images_dir}/{row['id']}.jpg"
#         try:
#             img = self.preprocess(Image.open(path).convert('RGB'))
#         except:
#             img = torch.zeros(3, 224, 224)
#         return img, row['id']

# print("Encoding all images with CLIP...")
# clip_dataset = FashionImageDataset(df_clean, images_dir, preprocess)
# clip_loader = DataLoader(clip_dataset, batch_size=256, shuffle=False, num_workers=2)

# all_embeddings = []
# all_ids = []

# with torch.no_grad():
#     for batch_idx, (imgs, ids) in enumerate(clip_loader):
#         imgs = imgs.to(device)
#         embs = model.encode_image(imgs)
#         embs = embs / embs.norm(dim=-1, keepdim=True)
#         all_embeddings.append(embs.cpu().numpy())
#         all_ids.extend(ids)
#         if batch_idx % 10 == 0:
#             print(f"  Batch {batch_idx}/{len(clip_loader)} done")

# all_embeddings = np.vstack(all_embeddings)
# all_ids = np.array(all_ids)

# np.save('/content/zara/clip_embeddings.npy', all_embeddings)
# np.save('/content/zara/clip_ids.npy', all_ids)
# print(f"\nDone! Embeddings shape: {all_embeddings.shape}")
# print("Saved to disk.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loaded embeddings: (25043, 512)


In [30]:
id_to_color = dict(zip(df_clean['id'], df_clean['baseColour']))
id_to_type = dict(zip(df_clean['id'], df_clean['articleType']))
id_to_row = df_clean.drop_duplicates(subset='id').set_index('id')

COMPLEMENTS = {
    'Tshirts':      ['Jeans', 'Shorts', 'Track Pants', 'Hoodies', 'Casual Shoes'],
    'Shirts':       ['Jeans', 'Shorts', 'Casual Shoes'],
    'Tops':         ['Jeans', 'Skirts', 'Shorts', 'Casual Shoes', 'Handbags'],
    'Hoodies':      ['Jeans', 'Track Pants', 'Shorts', 'Casual Shoes'],
    'Sweatshirts':  ['Jeans', 'Track Pants', 'Shorts', 'Casual Shoes'],
    'Sweaters':     ['Jeans', 'Skirts', 'Casual Shoes'],
    'Cardigans':    ['Jeans', 'Skirts', 'Casual Shoes'],
    'Knitwear':     ['Jeans', 'Skirts', 'Casual Shoes'],
    'Jackets':      ['Jeans', 'Tshirts', 'Casual Shoes'],
    'Coats':        ['Jeans', 'Sweaters', 'Casual Shoes'],
    'Blazers':      ['Jeans', 'Tshirts', 'Casual Shoes'],
    'Jeans':        ['Tshirts', 'Shirts', 'Hoodies', 'Sweaters', 'Casual Shoes', 'Jackets'],
    'Trousers':     ['Tshirts', 'Shirts', 'Blazers', 'Sweaters', 'Casual Shoes'],
    'Shorts':       ['Tshirts', 'Shirts', 'Hoodies', 'Casual Shoes'],
    'Track Pants':  ['Tshirts', 'Hoodies', 'Sweatshirts', 'Casual Shoes'],
    'Skirts':       ['Tops', 'Tshirts', 'Sweaters', 'Casual Shoes', 'Handbags'],
    'Dresses':      ['Casual Shoes', 'Handbags', 'Jackets', 'Cardigans'],
    'Bodysuits':    ['Jeans', 'Skirts', 'Casual Shoes'],
    'Casual Shoes': ['Jeans', 'Tshirts', 'Shirts', 'Shorts', 'Hoodies', 'Sweatshirts'],
    'Handbags':     ['Dresses', 'Tops', 'Jeans', 'Skirts'],
    'Skirts':       ['Tops', 'Tshirts', 'Sweaters', 'Casual Shoes', 'Handbags'],
}

COMPLEMENT_BONUS = 0.4

TEXT_TYPE_KEYWORDS = {
    'Jeans': ['jeans', 'denim'],
    'Tshirts': ['tshirt', 't-shirt', 'tee'],
    'Shirts': ['shirt', 'button'],
    'Casual Shoes': ['sneaker', 'casual shoe', 'loafer'],
    'Sports Shoes': ['sport shoe', 'running shoe', 'trainer', 'sneakers'],
    'Watches': ['watch'],
    'Tops': ['top', 'blouse'],
    'Heels': ['heel', 'stiletto', 'pump'],
    'Handbags': ['bag', 'handbag', 'purse'],
    'Kurtas': ['kurta', 'ethnic top'],
    'Formal Shoes': ['formal shoe', 'oxford', 'derby'],
    'Trousers': ['trouser', 'chino', 'slacks'],
    'Blazers': ['blazer', 'jacket', 'suit'],
    'Leggings': ['legging'],
    'Dresses': ['dress', 'gown'],
    'Shorts': ['short'],
    'Track Pants': ['track pant', 'sweatpant', 'jogger'],
    'Caps': ['cap', 'hat', 'beanie'],
    'Socks': ['sock'],
    'Belts': ['belt'],
    'Sandals': ['sandal', 'flip flop', 'slipper'],
}

ITEM_TYPE_LABELS = [
    'a t-shirt or tshirt',
    'a dress shirt or button-up shirt',
    'a jacket or coat',
    'a pair of jeans or trousers',
    'a pair of shorts',
    'a hoodie or sweatshirt',
    'a sweater or cardigan',
    'a blazer or suit jacket',
    'a dress, shirt dress, or midi dress',
    'a skirt',
    'a pair of shoes or sneakers',
    'a top or blouse',
    'a pair of track pants or joggers',
]

LABEL_TO_TYPE = {
    'a t-shirt or tshirt': 'Tshirts',
    'a dress shirt or button-up shirt': 'Shirts',
    'a jacket or coat': 'Jackets',
    'a pair of jeans or trousers': 'Jeans',
    'a pair of shorts': 'Shorts',
    'a hoodie or sweatshirt': 'Hoodies',
    'a sweater or cardigan': 'Sweaters',
    'a blazer or suit jacket': 'Blazers',
    'a dress, shirt dress, or midi dress': 'Dresses',
    'a skirt': 'Skirts',
    'a pair of shoes or sneakers': 'Casual Shoes',
    'a top or blouse': 'Tops',
    'a pair of track pants or joggers': 'Track Pants',
}

def detect_item_type_clip(pil_image):
    text_tokens = clip.tokenize(ITEM_TYPE_LABELS).to(device)
    image_tensor = preprocess(pil_image.convert('RGB')).unsqueeze(0).to(device)
    with torch.no_grad():
        image_features = model.encode_image(image_tensor)
        text_features = model.encode_text(text_tokens)
        image_features /= image_features.norm(dim=-1, keepdim=True)
        text_features /= text_features.norm(dim=-1, keepdim=True)
        similarity = (image_features @ text_features.T).softmax(dim=-1)
        probs = similarity[0].cpu().numpy()
    best_idx = probs.argmax()
    detected = ITEM_TYPE_LABELS[best_idx]
    print(f"CLIP type detection: {detected} ({probs[best_idx]:.3f})")
    return LABEL_TO_TYPE[detected]

def detect_item_type(query_emb, top_n=50):
    sims = all_embeddings @ query_emb
    top_indices = np.argsort(sims)[::-1][:top_n]
    top_ids = all_ids[top_indices]
    type_counts = {}
    for id_ in top_ids:
        atype = id_to_type.get(str(id_))
        if atype:
            type_counts[atype] = type_counts.get(atype, 0) + 1
    if not type_counts:
        return None
    sorted_types = sorted(type_counts.items(), key=lambda x: x[1], reverse=True)
    print(f"Type votes: {sorted_types[:5]}")
    return sorted_types[0][0]

def get_item_color(item_id):
    return id_to_color.get(str(item_id), None)

def get_query_color(query_emb, top_n=10):
    sims = all_embeddings @ query_emb
    top_indices = np.argsort(sims)[::-1][:top_n]
    top_ids = all_ids[top_indices]
    color_counts = {}
    for id_ in top_ids:
        color = id_to_color.get(str(id_))
        if color and color != 'Unknown':
            color_counts[color] = color_counts.get(color, 0) + 1
    if not color_counts:
        return None
    return max(color_counts, key=color_counts.get)

COLOR_LABELS = [
    'black', 'white', 'grey', 'navy blue', 'blue', 'brown',
    'beige', 'red', 'green', 'olive', 'pink', 'purple',
    'yellow', 'orange', 'cream', 'maroon'
]

def detect_color_from_image(pil_image):
    prompts = [f"a {color} colored clothing item" for color in COLOR_LABELS]
    text_tokens = clip.tokenize(prompts).to(device)
    image_tensor = preprocess(pil_image.convert('RGB')).unsqueeze(0).to(device)

    with torch.no_grad():
        image_features = model.encode_image(image_tensor)
        text_features = model.encode_text(text_tokens)
        image_features /= image_features.norm(dim=-1, keepdim=True)
        text_features /= text_features.norm(dim=-1, keepdim=True)
        similarity = (image_features @ text_features.T).softmax(dim=-1)
        probs = similarity[0].cpu().numpy()

    best_idx = probs.argmax()
    return COLOR_LABELS[best_idx]

def detect_type_from_text(text_query):
    text_lower = text_query.lower()
    for article_type, keywords in TEXT_TYPE_KEYWORDS.items():
        for keyword in keywords:
            if keyword in text_lower:
                return article_type
    return None

def recommend(uploaded_image, style, gender, top_k=5):
    img_tensor = preprocess(uploaded_image.convert('RGB')).unsqueeze(0).to(device)
    with torch.no_grad():
        query_emb = model.encode_image(img_tensor)
        query_emb = query_emb / query_emb.norm(dim=-1, keepdim=True)
        query_emb = query_emb.cpu().numpy().squeeze()

    query_type = detect_item_type_clip(uploaded_image)
    complements = COMPLEMENTS.get(query_type, [])
    query_color = get_query_color(query_emb)
    harmonious_colors = COLOR_HARMONY.get(query_color, [])

    style_mask = df_clean['usage'] == style
    gender_mask = df_clean['gender'].isin([gender])
    type_mask = df_clean['articleType'] != query_type

    if complements:
        candidate_mask = style_mask & gender_mask & df_clean['articleType'].isin(complements) & type_mask
    else:
        candidate_mask = style_mask & gender_mask & type_mask

    candidate_ids_series = df_clean[candidate_mask]['id']
    candidate_id_set = set(candidate_ids_series.values)

    indices = np.where(np.isin(all_ids, list(candidate_id_set)))[0]
    candidate_embeddings = all_embeddings[indices]
    candidate_ids = all_ids[indices]

    sims = candidate_embeddings @ query_emb
    color_bonuses = np.array([
        COLOR_HARMONY_BONUS if id_to_color.get(str(id_)) in harmonious_colors else 0.0
        for id_ in candidate_ids
    ])
    scores = sims + color_bonuses
    sorted_indices = np.argsort(scores)[::-1]

    seen_types = {}
    result_ids = []
    for idx in sorted_indices:
        if len(result_ids) == top_k:
            break
        id_ = candidate_ids[idx]
        atype = id_to_type.get(str(id_))
        if not atype or atype == query_type:
            continue
        if seen_types.get(atype, 0) < 1:
            result_ids.append(id_)
            seen_types[atype] = 1

    if len(result_ids) < top_k:
        checked = 0
        for idx in sorted_indices:
            if len(result_ids) == top_k or checked > 5000:
                break
            checked += 1
            id_ = candidate_ids[idx]
            if id_ in result_ids:
                continue
            atype = id_to_type.get(str(id_))
            if not atype or atype == query_type:
                continue
            if seen_types.get(atype, 0) < 2:
                result_ids.append(id_)
                seen_types[atype] = seen_types.get(atype, 0) + 1

    result_images = []
    result_details = []
    for id_ in result_ids:
        path = f"{images_dir}/{str(id_)}.jpg"
        img = Image.open(path).convert('RGB')
        result_images.append(img)
        row = id_to_row.loc[str(id_)]
        detail = f"{str(row['productDisplayName'])}\n{str(row['articleType'])} | {str(row.get('price', ''))} {str(row.get('currency', ''))}"
        result_details.append(detail)

    return result_images, result_details, query_type

def text_search(text_query, style, gender, top_k=5):
    text_token = clip.tokenize([text_query]).to(device)
    with torch.no_grad():
        text_emb = model.encode_text(text_token)
        text_emb = text_emb / text_emb.norm(dim=-1, keepdim=True)
        text_emb = text_emb.cpu().numpy().squeeze()

    query_type = detect_type_from_text(text_query)
    complements = COMPLEMENTS.get(query_type, [])

    style_mask = df_clean['usage'] == style
    gender_mask = df_clean['gender'].isin([gender])
    type_mask = df_clean['articleType'] != query_type if query_type else pd.Series([True]*len(df_clean))

    if complements:
        candidate_mask = style_mask & gender_mask & df_clean['articleType'].isin(complements) & type_mask
    else:
        candidate_mask = style_mask & gender_mask & type_mask

    candidate_id_set = set(df_clean[candidate_mask]['id'].values)
    indices = np.where(np.isin(all_ids, list(candidate_id_set)))[0]
    candidate_embeddings = all_embeddings[indices]
    candidate_ids = all_ids[indices]

    sims = candidate_embeddings @ text_emb
    sorted_indices = np.argsort(sims)[::-1]

    seen_types = {}
    result_ids = []
    for idx in sorted_indices:
        if len(result_ids) == top_k:
            break
        id_ = candidate_ids[idx]
        atype = id_to_type.get(str(id_))
        if not atype or atype == query_type:
            continue
        if seen_types.get(atype, 0) < 1:
            result_ids.append(id_)
            seen_types[atype] = 1

    if len(result_ids) < top_k:
        checked = 0
        for idx in sorted_indices:
            if len(result_ids) == top_k or checked > 5000:
                break
            checked += 1
            id_ = candidate_ids[idx]
            if id_ in result_ids:
                continue
            atype = id_to_type.get(str(id_))
            if not atype or atype == query_type:
                continue
            if seen_types.get(atype, 0) < 2:
                result_ids.append(id_)
                seen_types[atype] = seen_types.get(atype, 0) + 1

    result_images = []
    result_details = []
    for id_ in result_ids:
        path = f"{images_dir}/{str(id_)}.jpg"
        img = Image.open(path).convert('RGB')
        result_images.append(img)
        row = id_to_row.loc[str(id_)]
        detail = f"{row['productDisplayName']}\n{row['articleType']} | {row.get('price', '')} {row.get('currency', '')}"
        result_details.append(detail)

    return result_images, result_details

In [31]:
from itertools import product as iterproduct

COLOR_HARMONY = {
    'Black':     ['White', 'Grey', 'Red', 'Blue', 'Navy Blue', 'Pink', 'Yellow', 'Green'],
    'White':     ['Black', 'Navy Blue', 'Blue', 'Grey', 'Brown', 'Beige', 'Pink', 'Red'],
    'Blue':      ['White', 'Grey', 'Brown', 'Beige', 'Navy Blue', 'Black'],
    'Navy Blue': ['White', 'Grey', 'Beige', 'Brown', 'Black', 'Red'],
    'Grey':      ['Black', 'White', 'Navy Blue', 'Blue', 'Pink', 'Red'],
    'Brown':     ['Beige', 'White', 'Cream', 'Blue', 'Navy Blue', 'Olive'],
    'Beige':     ['Brown', 'White', 'Navy Blue', 'Olive', 'Black'],
    'Red':       ['Black', 'White', 'Navy Blue', 'Grey'],
    'Green':     ['Brown', 'Beige', 'White', 'Black', 'Olive'],
    'Olive':     ['Brown', 'Beige', 'White', 'Black', 'Navy Blue'],
    'Pink':      ['White', 'Grey', 'Black', 'Navy Blue', 'Beige'],
    'Purple':    ['White', 'Grey', 'Black', 'Beige'],
    'Yellow':    ['Black', 'White', 'Navy Blue', 'Grey'],
    'Orange':    ['Black', 'White', 'Navy Blue', 'Brown'],
    'Maroon':    ['White', 'Beige', 'Black', 'Grey'],
    'Cream':     ['Brown', 'Beige', 'Black', 'Navy Blue'],
    'Gold':      ['Black', 'White', 'Navy Blue', 'Brown'],
    'Silver':    ['Black', 'White', 'Grey', 'Navy Blue'],
}

COLOR_HARMONY_BONUS = 0.1


def build_shop_html(query_type=None, query_color=None, style=None, gender=None, complements=None):
    if not query_type:
        return ""

    MENS_ITEMS = ['Jeans', 'Trousers', 'Shorts', 'Track Pants', 'Tshirts', 'Shirts',
                  'Hoodies', 'Sweatshirts', 'Sweaters', 'Cardigans', 'Jackets',
                  'Coats', 'Blazers', 'Knitwear', 'Casual Shoes', 'Handbags']

    WOMENS_ITEMS = ['Jeans', 'Trousers', 'Shorts', 'Track Pants', 'Tshirts', 'Shirts',
                    'Hoodies', 'Sweatshirts', 'Sweaters', 'Cardigans', 'Jackets',
                    'Coats', 'Blazers', 'Knitwear', 'Casual Shoes', 'Handbags',
                    'Skirts', 'Dresses', 'Tops', 'Bodysuits']

    if complements:
        search_items = complements[:4]
    else:
        search_items = [query_type]

    if gender == 'Men':
        search_items = [i for i in search_items if i in MENS_ITEMS]
    elif gender == 'Women':
        search_items = [i for i in search_items if i in WOMENS_ITEMS]

    if not search_items:
        search_items = [query_type]

    harmonious_colors = COLOR_HARMONY.get(query_color.title() if query_color else '', [])
    top_colors = harmonious_colors[:2] if harmonious_colors else [query_color or '']

    html = f"""
    <div style='margin-top: 24px; font-family: sans-serif;
                border-top: 2px solid #f0f0f0; padding-top: 20px;'>
        <h3 style='color: #333; margin-bottom: 12px;'>🛍️ Shop Items That Go With Your {query_type}</h3>
        <div style='display: flex; gap: 12px; flex-wrap: wrap;'>
    """

    for i, item in enumerate(search_items):
        item_color = top_colors[i % len(top_colors)]

        search_term = f"{item_color} {item} {gender or 'men'} {style or ''}".strip()
        encoded = search_term.replace(' ', '+')
        google_url = f"https://www.google.com/search?q={encoded}&tbm=shop"
        asos_url = f"https://www.asos.com/search/?q={encoded}"

        html += f"""
        <div style='margin-bottom: 12px; width: 100%;'>
            <p style='margin: 0 0 8px 0; font-size: 13px; color: #333; font-weight: bold;'>
                {item_color.title()} {item}
            </p>
            <div style='display: flex; gap: 8px;'>
                <a href='{google_url}' target='_blank'
                   style='padding: 8px 16px; background: #4285f4; color: white;
                          border-radius: 6px; text-decoration: none; font-weight: bold;
                          font-size: 13px;'>
                   🔍 Google Shopping
                </a>
                <a href='{asos_url}' target='_blank'
                   style='padding: 8px 16px; background: #e91e8c; color: white;
                          border-radius: 6px; text-decoration: none; font-weight: bold;
                          font-size: 13px;'>
                   👗 ASOS
                </a>
            </div>
        </div>
        """

    html += "</div></div>"
    return html

In [32]:
import gradio as gr
from itertools import product as iterproduct
import base64
from io import BytesIO
import requests

RAPIDAPI_KEY = "df2fcf85a8mshabd473b909c5634p14a7fdjsndb7a4dbbe3c8"

CLOTHING_LABELS = [
    "a clothing item such as a shirt, pants, dress, jacket, or coat",
    "a person wearing an outfit, clothes, or fashion items",
    "footwear such as shoes, boots, sneakers, or sandals",
    "a fashion accessory such as a bag, watch, belt, or hat",
    "a flat lay product photo of a clothing item on white background",
    "a t-shirt, tshirt, or top laid flat or on a hanger",
    "pencils, pens, stationery, or office supplies",
    "furniture, a window, or a building",
    "food, a drink, or a meal",
    "a face or portrait photo with no visible clothing",
    "a landscape, nature, or outdoor scenery",
    "a toy, costume, or cartoon character",
    "a body part, hand, or decoration",
    "colorful objects, art supplies, or craft materials",
    "electronics, a phone, laptop, or gadget",
    "a car, vehicle, or transportation",
    "an animal or pet"
]

OUTFIT_ESSENTIALS = [
    ['Tshirts', 'Shirts', 'Tops', 'Kurtas', 'Blazers'],
    ['Jeans', 'Trousers', 'Track Pants', 'Shorts', 'Leggings', 'Dresses'],
    ['Casual Shoes', 'Sports Shoes', 'Formal Shoes', 'Heels', 'Sandals', 'Flats'],
]

def is_clothing_image(pil_image):
    text_tokens = clip.tokenize(CLOTHING_LABELS).to(device)
    image_tensor = preprocess(pil_image.convert('RGB')).unsqueeze(0).to(device)
    with torch.no_grad():
        image_features = model.encode_image(image_tensor)
        text_features = model.encode_text(text_tokens)
        image_features /= image_features.norm(dim=-1, keepdim=True)
        text_features /= text_features.norm(dim=-1, keepdim=True)
        similarity = (image_features @ text_features.T).softmax(dim=-1)
        probs = similarity[0].cpu().numpy()
    fashion_confidence = float(max(probs[:6]))
    non_fashion_confidence = float(max(probs[6:]))
    return fashion_confidence > non_fashion_confidence, fashion_confidence

def encode_pil_image(pil_image):
    img_tensor = preprocess(pil_image.convert('RGB')).unsqueeze(0).to(device)
    with torch.no_grad():
        emb = model.encode_image(img_tensor)
        emb = emb / emb.norm(dim=-1, keepdim=True)
    return emb.cpu().numpy().squeeze()

def score_outfit(embeddings):
    n = len(embeddings)
    if n < 2:
        return 0.0
    total, count = 0, 0
    for i in range(n):
        for j in range(i+1, n):
            total += np.dot(embeddings[i], embeddings[j])
            count += 1
    return total / count

def build_wardrobe_outfits(images):
    wardrobe = []
    for img in images:
        if img is None:
            continue
        if isinstance(img, dict):
            img = img.get("image") or img.get("data")
        if isinstance(img, tuple):
            img = img[0]
        if isinstance(img, str):
            pil_img = Image.open(img).convert('RGB')
        else:
            pil_img = Image.fromarray(img).convert('RGB')
        is_clothing, _ = is_clothing_image(pil_img)
        if not is_clothing:
            continue
        emb = encode_pil_image(pil_img)
        item_type = detect_item_type(emb)
        wardrobe.append({'image': pil_img, 'embedding': emb, 'type': item_type})

    if len(wardrobe) < 2:
        return None, "Please upload at least 2 clothing items."

    essential_groups = {}
    for item in wardrobe:
        for i, category_list in enumerate(OUTFIT_ESSENTIALS):
            if item['type'] in category_list:
                key = i
                if key not in essential_groups:
                    essential_groups[key] = []
                essential_groups[key].append(item)
                break

    if len(essential_groups) < 2:
        return None, "Please upload items of different types (e.g. a top + pants + shoes)."

    type_lists = list(essential_groups.values())
    all_combos = list(iterproduct(*type_lists))

    scored = []
    for combo in all_combos:
        embs = [item['embedding'] for item in combo]
        score = score_outfit(embs)
        scored.append((score, combo))

    scored.sort(key=lambda x: x[0], reverse=True)
    return scored[:10], None

def pil_to_base64(pil_img):
    buffer = BytesIO()
    pil_img.thumbnail((200, 300))
    pil_img.save(buffer, format='JPEG')
    encoded = base64.b64encode(buffer.getvalue()).decode()
    return f"data:image/jpeg;base64,{encoded}"

def search_asos(query, limit=4):
    url = "https://asos2.p.rapidapi.com/products/v2/list"
    headers = {
        "x-rapidapi-host": "asos2.p.rapidapi.com",
        "x-rapidapi-key": RAPIDAPI_KEY
    }
    params = {
        "store": "US",
        "offset": "0",
        "limit": str(limit),
        "country": "US",
        "sort": "freshness",
        "currency": "USD",
        "sizeSchema": "US",
        "lang": "en-US",
        "q": query
    }
    try:
        response = requests.get(url, headers=headers, params=params)
        data = response.json()
        return data.get('products', [])
    except:
        return []

def build_asos_html(products):
    if not products:
        return ""

    html = """
    <div style='margin-top: 24px; font-family: sans-serif;
                border-top: 2px solid #f0f0f0; padding-top: 20px;'>
        <h3 style='color: #333; margin-bottom: 16px;'>🛍️ Shop This Look on ASOS</h3>
        <div style='display: flex; gap: 16px; flex-wrap: wrap;'>
    """

    for p in products:
        name = p.get('name', '')
        price = p.get('price', {}).get('current', {}).get('text', 'N/A')
        image_url = p.get('imageUrl', '')
        if image_url and not image_url.startswith('http'):
            image_url = 'https://' + image_url
        product_url = p.get('url', '')
        if product_url and not product_url.startswith('http'):
            product_url = 'https://www.asos.com/' + product_url
        brand = p.get('brandName', '')

        html += f"""
        <div style='width: 160px; text-align: center;'>
            <a href='{product_url}' target='_blank' style='text-decoration: none; color: inherit;'>
                <img src='{image_url}'
                     style='width: 160px; height: 200px; object-fit: cover;
                            border-radius: 8px; border: 1px solid #ddd;'/>
                <p style='margin: 6px 0 2px 0; font-size: 11px; color: #888;'>{brand}</p>
                <p style='margin: 0 0 4px 0; font-size: 12px; color: #333;
                          font-weight: 500; line-height: 1.3;'>{name[:45]}...</p>
                <p style='margin: 0; font-size: 13px; color: #e91e8c;
                          font-weight: bold;'>{price}</p>
                <p style='margin: 4px 0 0 0; font-size: 11px; color: #666;
                          text-decoration: underline;'>View on ASOS →</p>
            </a>
        </div>
        """

    html += "</div></div>"
    return html

def fitmatch_wardrobe_from_state(wardrobe_state, style):
    if not wardrobe_state or len(wardrobe_state) < 2:
        return "<p>Add at least 2 items to your wardrobe first.</p>", ""

    scored, error = build_wardrobe_outfits(wardrobe_state)
    if error is not None:
        return f"<p>{error}</p>", ""

    html = "<div style='font-family: sans-serif;'>"
    for outfit_idx, (score, combo) in enumerate(scored):
        html += f"""
        <div style='margin-bottom: 24px; padding: 16px;
                    border: 1px solid #e0e0e0; border-radius: 12px;
                    background: #fafafa;'>
            <h3 style='margin: 0 0 12px 0; color: #333;'>
                Outfit {outfit_idx + 1}
                <span style='font-size: 14px; color: #888; font-weight: normal;'>
                    — Compatibility Score: {score:.3f}
                </span>
            </h3>
            <div style='display: flex; gap: 12px; flex-wrap: wrap;'>
        """
        for item in combo:
            b64 = pil_to_base64(item['image'])
            html += f"""
                <div style='text-align: center;'>
                    <img src='{b64}' style='height: 180px; width: auto;
                         border-radius: 8px; object-fit: cover;
                         border: 1px solid #ddd;'/>
                    <p style='margin: 6px 0 0 0; font-size: 12px;
                              color: #555;'>{item['type']}</p>
                </div>
            """
        html += "</div></div>"
    html += "</div>"

    label = f"Found {len(scored)} outfit combinations ranked by compatibility score."
    return html, label

def fitmatch_recommend(image, style, gender):
    if image is None:
        return None, "Please upload an image.", ""

    pil_image = Image.fromarray(image)
    is_clothing, confidence = is_clothing_image(pil_image)
    if not is_clothing:
        return None, "⚠️ This doesn't look like a clothing or fashion item. Please upload a photo of a clothing item such as a shirt, jeans, shoes, or an accessory.", ""

    results, details, detected_type = recommend(pil_image, style=style, gender=gender, top_k=5)
    gallery_items = [(img, detail) for img, detail in zip(results, details)]
    label = f"Detected item type: {detected_type or 'Unknown'} | Style: {style} | Gender: {gender}"

    img_tensor = preprocess(pil_image.convert('RGB')).unsqueeze(0).to(device)
    with torch.no_grad():
        query_emb = model.encode_image(img_tensor)
        query_emb = query_emb / query_emb.norm(dim=-1, keepdim=True)
        query_emb = query_emb.cpu().numpy().squeeze()
    query_color = detect_color_from_image(pil_image)
    harmonious_colors = COLOR_HARMONY.get(query_color.title(), [])
    search_color = harmonious_colors[0] if harmonious_colors else query_color
    complements = COMPLEMENTS.get(detected_type, [])

    shop_html = build_shop_html(detected_type, search_color, style, gender, complements)

    return gallery_items, label, shop_html

def fitmatch_text_search(text_query, style, gender):
    if not text_query.strip():
        return None, "Please enter a search query."
    results, details = text_search(text_query, style=style, gender=gender, top_k=5)
    gallery_items = [(img, detail) for img, detail in zip(results, details)]
    label = f"Search: '{text_query}' | Style: {style} | Gender: {gender}"
    return gallery_items, label

def _wardrobe_label(state):
    n = len(state) if state else 0
    return f"{n} item{'s' if n != 1 else ''} saved in wardrobe this session."

def add_to_wardrobe(new_images, wardrobe_state):
    if new_images is None or len(new_images) == 0:
        return wardrobe_state, _wardrobe_label(wardrobe_state)
    wardrobe_state = wardrobe_state or []
    wardrobe_state = wardrobe_state + list(new_images)
    return wardrobe_state, _wardrobe_label(wardrobe_state)

def clear_wardrobe():
    return [], "0 items saved in wardrobe this session."

with gr.Blocks(title="FitMatch — AI Outfit Recommender") as demo:
    gr.Markdown("# 👗 FitMatch — AI Outfit Recommender")
    gr.Markdown("Powered by CLIP — upload a clothing item or describe what you're looking for.")

    with gr.Tabs():
        with gr.Tab("📷 Upload Item"):
            gr.Markdown("### Upload a clothing item to get compatible outfit recommendations")
            with gr.Row():
                with gr.Column(scale=1):
                    image_input = gr.Image(label="Upload Clothing Item")
                    style_input = gr.Dropdown(
                        choices=['Casual', 'Formal'],
                        value='Casual',
                        label="Style"
                    )
                    gender_input = gr.Dropdown(
                        choices=['Men', 'Women'],
                        value='Men',
                        label="Gender"
                    )
                    submit_btn = gr.Button("Find Matches 🔍", variant="primary")
                with gr.Column(scale=2):
                    gallery_output = gr.Gallery(
                        label="Recommended Items",
                        columns=5,
                        height=400
                    )
                    label_output = gr.Textbox(label="Detection Info")
                    asos_output = gr.HTML(label="Shop This Look")

            submit_btn.click(
                fn=fitmatch_recommend,
                inputs=[image_input, style_input, gender_input],
                outputs=[gallery_output, label_output, asos_output]
            )

        with gr.Tab("🔍 Text Search"):
            gr.Markdown("### Describe what you're looking for in natural language")
            gr.Markdown("Examples: *'blue jeans'*, *'formal black shoes'*, *'red dress'*")
            with gr.Row():
                with gr.Column(scale=1):
                    text_input = gr.Textbox(
                        label="Describe what you're looking for",
                        placeholder="e.g. blue casual jeans"
                    )
                    style_input_text = gr.Dropdown(
                        choices=['Casual', 'Formal'],
                        value='Casual',
                        label="Style"
                    )
                    gender_input_text = gr.Dropdown(
                        choices=['Men', 'Women'],
                        value='Men',
                        label="Gender"
                    )
                    search_btn = gr.Button("Search 🔍", variant="primary")
                with gr.Column(scale=2):
                    gallery_output_text = gr.Gallery(
                        label="Search Results",
                        columns=5,
                        height=400
                    )
                    label_output_text = gr.Textbox(label="Search Info")

            search_btn.click(
                fn=fitmatch_text_search,
                inputs=[text_input, style_input_text, gender_input_text],
                outputs=[gallery_output_text, label_output_text]
            )

        with gr.Tab("👗 My Wardrobe"):
            gr.Markdown("### Build your wardrobe across multiple uploads")
            gr.Markdown("Upload items in batches — they stay saved for the session. Then generate outfit combos from everything you've added.")

            wardrobe_state = gr.State([])

            with gr.Row():
                with gr.Column(scale=1):
                    wardrobe_upload = gr.Gallery(
                        label="Upload new items",
                        columns=3,
                        height=220,
                        type="numpy",
                        interactive=True
                    )
                    wardrobe_style = gr.Dropdown(
                        choices=['Casual', 'Formal'],
                        value='Casual',
                        label="Style"
                    )
                    with gr.Row():
                        add_btn = gr.Button("Add to Wardrobe ➕", variant="secondary")
                        clear_btn = gr.Button("Clear Wardrobe 🗑️")
                    wardrobe_counter = gr.Textbox(
                        label="Saved items",
                        value="0 items saved in wardrobe this session.",
                        interactive=False
                    )
                    generate_btn = gr.Button("Create Outfits 👗", variant="primary")

                with gr.Column(scale=2):
                    wardrobe_output = gr.HTML(label="Outfit Combinations")
                    wardrobe_label = gr.Textbox(label="Results Info")

            add_btn.click(
                fn=add_to_wardrobe,
                inputs=[wardrobe_upload, wardrobe_state],
                outputs=[wardrobe_state, wardrobe_counter]
            )
            clear_btn.click(
                fn=clear_wardrobe,
                outputs=[wardrobe_state, wardrobe_counter]
            )
            generate_btn.click(
                fn=fitmatch_wardrobe_from_state,
                inputs=[wardrobe_state, wardrobe_style],
                outputs=[wardrobe_output, wardrobe_label]
            )

demo.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://39c5f5dba2a79e8714.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


CLIP type detection: a dress shirt or button-up shirt (0.083)
Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://39c5f5dba2a79e8714.gradio.live


In [27]:
df[df['productDisplayName'].str.contains('Denim Shirt', case=False)]['productDisplayName'].unique()

array(['Denim Shirt', 'Z1975 Denim Shirt',
       'Dégradé Denim Shirt Zw Collection'], dtype=object)